<a href="https://colab.research.google.com/github/Adhiaris/Grokking-Deep-Learning/blob/main/chapter_06_backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6: Building Your First Deep Neural Network
### Introduction to Backpropagation

This chapter introduces the **streetlight problem** as a real dataset, trains a single-layer network on it, then extends to a deep (two-layer) network using backpropagation.

## 1. The Streetlight Problem

You observe streetlight patterns and whether people walk or stop. The goal is to train a network to predict the correct action from the light pattern. The true signal is the middle light.

In [ ]:
import numpy as np

streetlights = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1],
    [0, 1, 1],
    [1, 0, 1],
])

walk_vs_stop = np.array([0, 1, 0, 1, 1, 0])

print("Streetlight patterns -> Walk/Stop labels")
print(f"{'Left':>6} {'Mid':>6} {'Right':>6}   {'Label'}")
print("-" * 35)
for i, (row, label) in enumerate(zip(streetlights, walk_vs_stop)):
    action = "WALK" if label == 1 else "STOP"
    print(f"{row[0]:>6} {row[1]:>6} {row[2]:>6}   {action}")

## 2. Single-Layer Network: Learning One Example

Start by training on just the first streetlight pattern using stochastic gradient descent.

In [ ]:
import numpy as np

weights = np.array([0.5, 0.48, -0.7])
alpha   = 0.1

streetlights = np.array([[1,0,1],[0,1,1],[0,0,1],[1,1,1],[0,1,1],[1,0,1]])
walk_vs_stop = np.array([0, 1, 0, 1, 1, 0])

input_val       = streetlights[0]
goal_prediction = walk_vs_stop[0]

print(f"{'Iter':<6} {'Prediction':>12} {'Error':>14}")
print("-" * 35)

for iteration in range(20):
    prediction = input_val.dot(weights)
    error      = (goal_prediction - prediction) ** 2
    delta      = prediction - goal_prediction
    weights   -= alpha * (input_val * delta)
    if iteration % 4 == 0:
        print(f"{iteration:<6} {prediction:>12.6f} {error:>14.8f}")

## 3. Stochastic Gradient Descent: Learning the Whole Dataset

Train on every streetlight one at a time, cycling through the full dataset multiple times. This is stochastic gradient descent.

In [ ]:
import numpy as np

weights = np.array([0.5, 0.48, -0.7])
alpha   = 0.1

streetlights = np.array([[1,0,1],[0,1,1],[0,0,1],[1,1,1],[0,1,1],[1,0,1]])
walk_vs_stop = np.array([0, 1, 0, 1, 1, 0])

print(f"{'Epoch':<8} {'Total Error':>14}")
print("-" * 25)

for iteration in range(40):
    total_error = 0
    for row_index in range(len(walk_vs_stop)):
        inp  = streetlights[row_index]
        goal = walk_vs_stop[row_index]
        pred  = inp.dot(weights)
        error = (goal - pred) ** 2
        total_error += error
        delta    = pred - goal
        weights -= alpha * (inp * delta)
    if iteration % 8 == 0:
        print(f"{iteration:<8} {total_error:>14.8f}")

print(f"\nFinal weights: {np.round(weights, 4)}")
print("Middle weight is highest — network found the correlation!")

## 4. Backpropagation: Learning in a Deep Network

A single-layer network fails when the correlation between input and output is indirect (not directly observable). A deep network creates intermediate representations. Backpropagation propagates error from the output layer back through the hidden layer to update all weights.

In [ ]:
import numpy as np

np.random.seed(1)

streetlights = np.array([[1,0,1],[0,1,1],[0,0,1],[1,1,1],[0,1,1],[1,0,1]], dtype=float)
walk_vs_stop = np.array([[0],[1],[0],[1],[1],[0]], dtype=float)

alpha   = 0.2
hidden_size = 4

weights_0_1 = 2 * np.random.rand(3, hidden_size) - 1
weights_1_2 = 2 * np.random.rand(hidden_size, 1) - 1

def relu(x):
    return (x > 0) * x

def relu_deriv(output):
    return output > 0

print(f"{'Epoch':<8} {'Error':>12}")
print("-" * 22)

for iteration in range(60):
    layer_2_error = 0
    for i in range(len(streetlights)):
        layer_0 = streetlights[i:i+1]
        layer_1 = relu(layer_0.dot(weights_0_1))
        layer_2 = layer_1.dot(weights_1_2)

        layer_2_error += np.sum((layer_2 - walk_vs_stop[i:i+1]) ** 2)

        layer_2_delta = layer_2 - walk_vs_stop[i:i+1]
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu_deriv(layer_1)

        weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)

    if iteration % 10 == 0:
        print(f"{iteration:<8} {layer_2_error:>12.6f}")

## 5. Key Concepts: Overfitting and Correlation

A network learns by finding correlation. If weights between correlated inputs and outputs receive consistent up-pressure and uncorrelated ones receive mixed pressure, the network zeros out noise and amplifies signal.

In [ ]:
import numpy as np

streetlights = np.array([[1,0,1],[0,1,1],[0,0,1],[1,1,1],[0,1,1],[1,0,1]], dtype=float)
walk_vs_stop = np.array([0, 1, 0, 1, 1, 0], dtype=float)

weights = np.array([0.5, 0.48, -0.7])
alpha   = 0.01

for _ in range(1000):
    for i in range(len(walk_vs_stop)):
        inp   = streetlights[i]
        goal  = walk_vs_stop[i]
        pred  = inp.dot(weights)
        delta = pred - goal
        weights -= alpha * inp * delta

print("Final weights after training:")
print(f"  Left  (no signal) : {weights[0]:.4f}")
print(f"  Middle (signal)   : {weights[1]:.4f}")
print(f"  Right  (no signal): {weights[2]:.4f}")
print("\nMiddle weight should be close to 1, others close to 0.")